# Sweep 11b — Labs Thesis: Lab Count Sweep

**Question:** How many labs do we need? Is there a saturation point? Do rare specialty labs (>200) add noise or signal?

**13 configs × 3 seeds = 39 runs ≈ 6–7 hours**

Backbone: naked + N labs, no copy, no notes, flat encoder. Only N varies.

| N | pkl | lab_dim |
|---|-----|---------|
| 5 | lab_vectors_5labs.pkl | 10 |
| 10 | lab_vectors_10labs.pkl | 20 |
| 20 | lab_vectors_20labs.pkl | 40 |
| 50 | lab_vectors_50labs.pkl | 100 |
| 100 | lab_vectors_100labs.pkl | 200 |
| 150 | lab_vectors_150labs.pkl | 300 |
| 200 | lab_vectors_200labs.pkl | 400 |
| 204 | lab_vectors_204_MAXlabs.pkl | 408 |
| 250 | lab_vectors_250labs.pkl | 500 |
| 300 | lab_vectors_300labs.pkl | 600 |
| 350 | lab_vectors_350labs.pkl | 700 |
| 400 | lab_vectors_400labs.pkl | 800 |
| 446 | lab_vectors_446_MAXlabs.pkl | 892 |

**Reference:** Run 11a first to get A0 (naked floor) and A1 (200 labs reference).

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

BACKBONE_MODEL = {
    "graph_encoder_type": "hgt_ehr_feat",
    "hgt_layers": 1,
    "pos_weight_cap": 15.0,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
)

# (n_labs, short_name, pkl_suffix)
LAB_COUNTS = [
    (5,   "n5",   "5labs"),
    (10,  "n10",  "10labs"),
    (20,  "n20",  "20labs"),
    (50,  "n50",  "50labs"),
    (100, "n100", "100labs"),
    (150, "n150", "150labs"),
    (200, "n200", "200labs"),
    (204, "n204", "204_MAXlabs"),
    (250, "n250", "250labs"),
    (300, "n300", "300labs"),
    (350, "n350", "350labs"),
    (400, "n400", "400labs"),
    (446, "n446", "446_MAXlabs"),
]

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_11b_count_sweep")
results_log = []

print(f"Lab count sweep: {len(LAB_COUNTS)} configs × {len(SEEDS)} seeds = {len(LAB_COUNTS)*len(SEEDS)} runs")

In [ ]:
# Smoke test — 3 representative counts at 1 epoch
import subprocess
from pathlib import Path

Path("smoke_s11b").mkdir(exist_ok=True)

SMOKE_COUNTS = [(5, "5labs"), (200, "200labs"), (446, "446_MAXlabs")]
smoke_results = []

for n, suffix in SMOKE_COUNTS:
    cfg_path = f"s11b_smoke_n{n}.yaml"
    make_config(cfg_path,
                model_overrides={"copy_mechanism": False, **BACKBONE_MODEL},
                preprocessing_overrides={"note_method": "none", "lab_dim": n*2},
                smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} "
           f"--lab_pkl data/processed/lab_vectors_{suffix}.pkl --num_labs {n} "
           f"--seed 42 --results_dir smoke_s11b/n{n}")
    print(f"SMOKE n={n}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (proc.stdout + proc.stderr).strip().split("\n")[-5:]
    for l in tail: print(l)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: n{n}")
    print(f">>> {status}\n")

for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")

In [ ]:
import subprocess, gc, torch

for n, short, pkl_suffix in LAB_COUNTS:
    cname = f"B_{short}"
    print(f"\n{'#'*70}\n# {cname} — {n} labs ({n*2}d features)\n{'#'*70}")
    cfg_path = f"s11b_{cname}.yaml"
    make_config(cfg_path,
                model_overrides={"copy_mechanism": False, **BACKBONE_MODEL},
                preprocessing_overrides={"note_method": "none", "lab_dim": n*2})

    for seed in SEEDS:
        run_name = f"{cname}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        log_path = run_dir / "training_log.txt"
        cmd = (
            f"python -u src/train.py --config {cfg_path} "
            f"{BACKBONE_FLAGS} "
            f"--lab_pkl data/processed/lab_vectors_{pkl_suffix}.pkl --num_labs {n} "
            f"--seed {seed} --results_dir {run_dir}"
        )
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nDone — {len(results_log)} runs | {sum(1 for r in results_log if 'SUCCESS' in r)} succeeded")

In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_11b_count_sweep")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    results.setdefault(run_name[:idx], []).append(d)

# Paste A0/A1 Jaccard from 11a results here after running 11a
JAC_A0_NAKED = 0.0   # <-- fill in from 11a A0 result
JAC_A1_200   = 0.0   # <-- fill in from 11a A1 result

print("\n" + "="*80)
print("SWEEP 11b — LAB COUNT SWEEP RESULTS")
print("="*80)
print(f"  Reference — A0 naked:  {JAC_A0_NAKED:.4f}  (from 11a)")
print(f"  Reference — A1 200labs:{JAC_A1_200:.4f}  (from 11a)")
print()
print(f"  {'N labs':<10} {'Jac mean±std':>16}  {'Δ naked':>8}  {'Δ 200labs':>10}  n")
print(f"  {'-'*10} {'-'*16}  {'-'*8}  {'-'*10}  -")

count_jacs = {}
for n, short, _ in LAB_COUNTS:
    cname = f"B_{short}"
    runs = results.get(cname, [])
    if not runs:
        print(f"  {n:<10} {'(missing)':>16}")
        continue
    jac = [get_metric(d, "jaccard") for d in runs]
    mean, std = float(np.mean(jac)), float(np.std(jac, ddof=1) if len(jac)>1 else 0)
    d_naked = f"{mean - JAC_A0_NAKED:+.4f}"
    d_200   = f"{mean - JAC_A1_200:+.4f}"
    print(f"  {n:<10} {mean:.4f}±{std:.4f}  {d_naked:>8}  {d_200:>10}  {len(jac)}")
    count_jacs[n] = mean

if count_jacs:
    best_n = max(count_jacs, key=count_jacs.get)
    print(f"\n  Best count: {best_n} labs → Jac={count_jacs[best_n]:.4f}")
    sorted_c = sorted(count_jacs.items())
    for i in range(1, len(sorted_c)):
        n1, j1 = sorted_c[i-1]
        n2, j2 = sorted_c[i]
        if abs(j2 - j1) < 0.001 and n1 >= 50:
            print(f"  Saturation around {n1} labs (Δ {n1}→{n2}: {j2-j1:+.4f})")
            break

In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_11b_count_sweep.zip"
rd = Path("experiment_reports/sweep_11b_count_sweep")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")): zf.write(p, p.relative_to(rd))
    for p in sorted(rd.rglob("training_log.txt")): zf.write(p, p.relative_to(rd))
print(f"Exported → {zip_name}")